# Minimum Birkhoff Decomposition

This notebook combines circuit sampling with classical integer optimisation.


## Workflow

Each iteration evaluates parameterised circuits. The cost function decodes each measured bitstring into a permutation combination and calls `solve_black_box` (CPLEX) on it, with calls dispatched over a thread pool.


In [ ]:
import json

import numpy as np
from birkhoff import encode_combination, run_birkhoff
from main import parse_instance
from qiskit.circuit.library import CZGate, RYGate

from divi.backends import MaestroSimulator
from divi.qprog.algorithms import GenericLayerAnsatz
from divi.qprog.optimizers import ScipyMethod, ScipyOptimizer

## How It Works

This example runs a circuit-sampling Birkhoff decomposition of a doubly stochastic matrix.  The cost function is a CPLEX integer program over measured bitstrings, not a Hamiltonian expectation value.

1. A standalone **`CircuitPipeline`** (`QiskitSpecStage → MeasurementStage(COUNTS) → ParameterBindingStage`) maps parameter sets to raw shot histograms.
2. A plain cost function decodes each bitstring into a permutation combination, calls **`solve_black_box`** (CPLEX) through a thread pool, with caching, and returns a scalar loss.
3. A Divi optimiser (`ScipyOptimizer` or `MonteCarloOptimizer`) iterates the cost function, up to `max_iterations`, via its public `optimize(...)` API.


## Pick a target matrix


In [ ]:
N_DIM = 3  # Matrix dimension (3 or 4)
K_COMBINATIONS = 2  # Number of permutations in the decomposition
INSTANCE_ID = "1"  # Problem instance (1-10)
MATRIX_TYPE = "sparse"
MAX_ITERATIONS = 10

mat_file = f"qbench_0{N_DIM}_{MATRIX_TYPE}.json"
perm_file = f"p{N_DIM}.dat"

with open(mat_file) as f:
    problem_data = json.load(f)
permutations = np.loadtxt(perm_file, dtype=int)

all_permutation_matrices = np.eye(N_DIM, dtype=int)[permutations - 1]

matrix, scale, _, _ = parse_instance(problem_data, INSTANCE_ID)

print(f"Matrix dimension: {N_DIM}×{N_DIM}")
print(f"Target decomposition: {K_COMBINATIONS} permutations")
print(f"Scale factor: {scale}")
print("\nTarget matrix (scaled):")
print(matrix)

---

## Drive the pipeline with COBYLA

Uses Divi's **GenericLayerAnsatz** with RY rotations and CZ entanglers in a brick layout.


In [ ]:
optimizer = ScipyOptimizer(ScipyMethod.COBYLA)
backend = MaestroSimulator(shots=5000)
ansatz = GenericLayerAnsatz([RYGate], entangler=CZGate, entangling_layout="brick")

print("Starting optimisation...")
result = run_birkhoff(
    matrix=matrix,
    scale=scale,
    k=K_COMBINATIONS,
    all_perms_matrix=all_permutation_matrices,
    backend=backend,
    optimizer=optimizer,
    max_iterations=MAX_ITERATIONS,
    ansatz=ansatz,
    n_layers=3,
)

## Read the decomposition off the histogram


In [ ]:
if result.combination is not None:
    total_shots = sum(result.final_histogram.values())
    bitstring_width = len(next(iter(result.final_histogram)))
    solution_integer = encode_combination(result.combination, K_COMBINATIONS)
    solution_bitstring = format(solution_integer, f"0{bitstring_width}b")
    solution_shots = result.final_histogram.get(solution_bitstring, 0)
    probability = (solution_shots / total_shots) * 100 if total_shots else 0.0

    print("Shots on the selected combination's bitstring:")
    print(
        f"  Bitstring '{solution_bitstring}' — {probability:.2f}% ({solution_shots}/{total_shots} shots)"
    )

    print("\nDecomposition (Found):")
    for perm_idx, weight in zip(result.combination, result.weights):
        if weight > 1e-6:
            print(f"  Permutation {perm_idx}: weight = {weight:.6f}")

    reconstructed = sum(
        w * all_permutation_matrices[p]
        for p, w in zip(result.combination, result.weights)
        if w > 1e-6
    )
    original_unscaled = matrix / scale
    error_norm = np.linalg.norm(original_unscaled - reconstructed, ord=2)
    print(f"\nReconstruction error (spectral norm): {error_norm:.6f}")
    print(f"Total circuits: {result.total_circuit_count}")
    print(f"\nOriginal (unscaled):\n{original_unscaled}")
    print(f"\nReconstructed:\n{reconstructed}")
else:
    print("No decomposition was obtained from the top measurement outcomes.")

---

## Further exploration

`N_DIM` can be set to 4.  To run on QoroService instead:

```python
from divi.backends import JobConfig, QoroService
qpu_backend = QoroService(job_config=JobConfig(shots=10_000))
```

Use the configured local backend to repeat the experiment with different instances.
